# Ordinal initialization ablation

Test the prespecified PC1, training endpoint contrast, and soft ordinal-code initializations against the **same** shared repeated-structure atlas objective. The model still integrates unknown physical-realization membership into its residual distribution; initialization must not turn true simulated fine position into supervision.

Keep the count representation, all fitting penalties, weak label strength, 60-iteration limit and tolerance fixed. The synthetic scenarios, seeds and query perturbations reproduce notebook 19. Record all solver outcomes and objectives. Only converged fits produce accepted query projections; do not choose an initializer from query recovery. Exact simulated coarse labels make this an optimistic test. Domain-coupling ambiguity is deliberately retained.

In [ ]:
from pathlib import Path
import hashlib,io,json,os,sys
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from IPython.display import display
REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pseudospace').is_dir())
sys.path.insert(0,str(REPO))
from pseudospace.repeated_structure_simulation import simulate_repeated_sections
from pseudospace.pseudostructure_atlas import fit_pseudostructure_atlas,project_pseudostructure_atlas
from pseudospace.count_representation import transform_count_representation
from pseudospace.repeated_atlas_baselines import fit_pc1_reference,project_pc1_reference,fit_segment_mixture,predict_segment_mixture
from pseudospace.stage_cache import cached_payload
from pseudospace.atlas_initialization import initialize_atlas_positions
from pseudospace.count_representation import fit_count_representation
from pseudospace.joint_repeated_atlas import fit_joint_atlas,project_joint_atlas
from pseudospace.count_rate_atlas import fit_count_rate_atlas
NOTEBOOK_LOGIC_VERSION='20.initialization_ablation.1'
SEEDS=(19,41,73)
OUT=Path(os.environ.get('PSEUDOSPACE_RESULTS_ROOT',str(REPO/'results'))).expanduser()/'repeated_structure_initialization_ablation'
OUT.mkdir(parents=True,exist_ok=True)
FIT_PARAMS=dict(n_components=15,seed=15,grid=np.linspace(0,1,101),roughness=.001,
    offset_penalty=4.,anatomy_strength=1.,anatomy_width=.1,max_iter=60,tol=1e-5,
    representation_theta=100.,count_theta=100.,rate_bandwidth=.08,min_effective=12.)
CONFIG=dict(version=NOTEBOOK_LOGIC_VERSION,seeds=SEEDS,n_train_per_specimen=500,
    n_query=400,spatial_strength=3.,nuisance_strength=3.,realization_sd=.1,
    partial_interval=[.35,.65],thin_fraction=.25,
    fit={**FIT_PARAMS,'grid':FIT_PARAMS['grid'].tolist()})
INITIALIZERS=('pc1','endpoint_contrast','soft_segment')
CONFIG['initializers']=INITIALIZERS
MODULES=['atlas_initialization.py','repeated_structure_simulation.py','pseudostructure_atlas.py','count_representation.py',
    'joint_repeated_atlas.py','count_rate_atlas.py','repeated_atlas_baselines.py']
CODE_DIGEST=hashlib.sha256(b''.join((REPO/'pseudospace'/f).read_bytes() for f in MODULES)).hexdigest()
(OUT/'manifest.json').write_text(json.dumps(dict(config=CONFIG,code_digest=CODE_DIGEST),indent=2))

## Training-only initializations

PC1 uses the frozen count representation and existing orientation/scaling. Endpoint contrast equally averages original-specimen S3 and S1 means before taking their difference; it uses training quantiles only. Soft segmentation predicts expected coarse ordinal code from training expression, without fixed anatomical transition locations or true fine positions. Both add coarse-label information to the starting point, while the fitting objective remains unchanged.

The solver is exposed directly here to record failed objectives without relaxing the public convergence guard. Accepted references use the same Gaussian projection and conditional count-rate components as the public interface.

In [ ]:
def predictors(reference,query,pc,mixture):
    projection=project_pseudostructure_atlas(reference,query['counts'],query['library'],
        query['gene_names'],query['structure_ids'])
    Y=transform_count_representation(query['counts'],query['library'],reference['representation'])
    coordinates={name:projection[name]['z_mean'] for name in ('gaussian','count')}
    coordinates['count_PC1']=project_pc1_reference(pc,Y)
    coordinates['soft_segment']=predict_segment_mixture(mixture,Y)['prediction'][:,0]
    return projection,coordinates

def evaluation_rows(query,projection,coordinates,key,original=None):
    rows=[]
    for method,z in coordinates.items():
        for segment in [None,0,1,2]:
            use=np.ones(len(z),bool) if segment is None else query['anatomy']==segment
            if use.sum()<8: continue
            truth=query['true_z'][use]
            varying=np.ptp(z[use])>0 and np.ptp(truth)>0
            row=dict(**key,method=method,segment='all' if segment is None else ['S1','S2','S3'][segment],
                n=int(use.sum()),spearman=float(spearmanr(z[use],truth).statistic) if varying else np.nan,
                gauge_mae=float(np.mean(np.abs(z[use]-truth))),
                z_q05=float(np.quantile(z[use],.05)),z_q95=float(np.quantile(z[use],.95)),
                endpoint_fraction=float(np.mean((z[use]<.01)|(z[use]>.99))))
            if method in ('gaussian','count'):
                row['conditional_entropy']=float(projection[method]['entropy'][use].mean())
                row['model_disagreement']=float(projection['coordinate_disagreement'][use].mean())
            if original is not None:
                row['paired_thinning_mae']=float(np.mean(np.abs(z[use]-original[method][use])))
            rows.append(row)
    return rows

def subset_query(query,use):
    return {key:value[use] if key!='gene_names' else value.copy() for key,value in query.items()}



In [ ]:
def run_initialization_ablation():
    score_rows,diagnostic_rows,history_rows,check_rows,failures=[],[],[],[],[]
    for seed in SEEDS:
        for scenario,mode,strength in [('spatial_only','independent',0.),
                ('independent_nuisance','independent',3.),('coupled_nuisance','coupled',3.)]:
            train=simulate_repeated_sections(seed=seed,n_per_specimen=500,
                specimens=('train_A','train_B'),nuisance_mode=mode,nuisance_strength=strength)
            Y,representation=fit_count_representation(train['counts'],train['library'],train['specimens'],
                n_components=15,theta=100.,seed=15)
            pc=fit_pc1_reference(Y,train['anatomy'])
            mixture=fit_segment_mixture(Y,train['anatomy'][:,None]/2.,train['anatomy'],train['specimens'],C=1.,seed=15)
            for initializer in INITIALIZERS:
                key=dict(seed=seed,training_scenario=scenario,initializer=initializer)
                try:
                    initial,initial_meta=initialize_atlas_positions(Y,train['anatomy'],train['specimens'],
                        method=initializer,seed=15)
                    gaussian=fit_joint_atlas(Y,train['specimens'],train['anatomy'],initial_z=initial,
                        grid=FIT_PARAMS['grid'],roughness=.001,offset_penalty=4.,anatomy_strength=1.,
                        anatomy_width=.1,max_iter=60,tol=1e-5)
                    diagnostic_rows.append(dict(**key,converged=bool(gaussian['converged']),
                        iterations=len(gaussian['history']),objective=gaussian['history'][-1]['objective'],
                        objective_change=gaussian['history'][-1]['objective_change'],
                        initial_position_rank=float(spearmanr(initial,train['true_z']).statistic),
                        initial_nuisance_abs_rank=float(abs(spearmanr(initial,train['nuisance_q']).statistic))))
                    history_rows.extend(dict(**key,**row) for row in gaussian['history'])
                    if not gaussian['converged']:
                        raise RuntimeError('joint Gaussian atlas did not converge')
                    train_z=np.empty(len(Y))
                    for specimen in np.unique(train['specimens']):
                        use=train['specimens']==specimen
                        train_z[use]=project_joint_atlas(gaussian,Y[use],offset=gaussian['offsets'][specimen])['z_mean']
                    rate=fit_count_rate_atlas(train['counts'],train['library'],train_z,train['specimens'],
                        grid=FIT_PARAMS['grid'],bandwidth=.08,theta=100.,min_effective=12.)
                    reference=dict(gaussian=gaussian,rate=rate,representation=representation,
                        gene_names=train['gene_names'],model_gene_mask=np.ones(40,bool))
                    query_modes=[mode,'reversed'] if mode=='coupled' else [mode]
                    for query_mode in query_modes:
                        for coverage,interval in [('full',(0.,1.)),('partial',(.35,.65))]:
                            query=simulate_repeated_sections(seed=seed+1000,n_per_specimen=400,
                                specimens=('unseen',),interval=interval,nuisance_mode=query_mode,nuisance_strength=strength)
                            projection,coordinates=predictors(reference,query,pc,mixture)
                            meta=dict(**key,query_mode=query_mode,coverage=coverage,measurement='original')
                            score_rows.extend(evaluation_rows(query,projection,coordinates,meta))
                            use=(query['true_z']>=.35)&(query['true_z']<=.65)
                            sub_projection,sub_coordinates=predictors(reference,subset_query(query,use),pc,mixture)
                            maximum=max(float(np.max(np.abs(sub_coordinates[m]-coordinates[m][use]))) for m in coordinates)
                            check_rows.append(dict(**key,query_mode=query_mode,coverage=coverage,subset_max_change=maximum))
                            rng=np.random.default_rng(seed+2000)
                            thinned={k:v.copy() for k,v in query.items()}
                            thinned['counts']=rng.binomial(query['counts'],.25)
                            thinned['library']=thinned['counts'].sum(axis=1)
                            if np.any(thinned['library']==0):raise ValueError('Empty thinned library')
                            thin_projection,thin_coordinates=predictors(reference,thinned,pc,mixture)
                            score_rows.extend(evaluation_rows(thinned,thin_projection,thin_coordinates,
                                {**meta,'measurement':'quarter_reads'},original=coordinates))
                    print('Completed initialization:',key,flush=True)
                except (ValueError,RuntimeError,np.linalg.LinAlgError) as exc:
                    failures.append(dict(**key,error=str(exc)))
                    print('Initialization failure:',failures[-1],flush=True)
    return {k:pd.DataFrame(v).to_json(orient='table') for k,v in dict(scores=score_rows,
        diagnostics=diagnostic_rows,history=history_rows,checks=check_rows,failures=failures).items()}

In [ ]:
payload=cached_payload('repeated_structure_initialization_ablation',run_initialization_ablation,
    root=OUT/'stage_cache',params=CONFIG,inputs={'seeds':np.array(SEEDS)},code=CODE_DIGEST)
results={key:pd.read_json(io.StringIO(str(value)),orient='table') for key,value in payload.items()}
for key,frame in results.items():frame.to_csv(OUT/(key+'.csv'),index=False)
# Guards run for cache hits too. Failures remain a reported outcome.
if len(results['diagnostics'])!=len(SEEDS)*3*len(INITIALIZERS):
    raise RuntimeError('Missing prespecified solver diagnostics')
if not results['checks'].empty and results['checks'].subset_max_change.max()>1e-10:
    raise RuntimeError('Query population composition changes frozen coordinates')
display(results['diagnostics'].pivot_table(index='training_scenario',columns='initializer',
    values=['converged','iterations','objective','initial_position_rank']))
display(results['failures'])

## Compare recovery and objective without selecting on query truth

Solver objectives only compare starts on the same training data and model. A better objective does not necessarily identify the anatomical process, and a nonconverged fit stays excluded. Summaries average three fixed simulation seeds, not independent kidney specimens. Reversed coupling tests an unmodeled domain change, not a claim that injury is always orthogonal to position.

In [ ]:
scores=results['scores']
if not scores.empty:
    display(scores.query("segment=='all' and measurement=='original' and method in ['gaussian','count']").pivot_table(
        index=['training_scenario','query_mode','coverage'],columns=['initializer','method'],values='spearman'))
    display(scores.query("segment!='all' and measurement=='original' and method=='gaussian'").pivot_table(
        index=['training_scenario','query_mode','coverage','segment'],columns='initializer',values='spearman'))
    display(scores.query("segment=='all' and measurement=='quarter_reads' and method in ['gaussian','count']").pivot_table(
        index=['training_scenario','query_mode','coverage'],columns=['initializer','method'],values='paired_thinning_mae'))
    import matplotlib.pyplot as plt
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    for ax,value,title in zip(axes,['objective','initial_position_rank'],['Training objective','Initial true-position rank']):
        for method in INITIALIZERS:
            panel=results['diagnostics'].query('initializer==@method')
            for j,scenario in enumerate(['spatial_only','independent_nuisance','coupled_nuisance']):
                rows=panel.query('training_scenario==@scenario')
                color=['C0','C1','C2'][INITIALIZERS.index(method)]
                for converged,marker in [(True,'o'),(False,'x')]:
                    selected=rows.loc[rows.converged==converged]
                    ax.scatter(np.full(len(selected),j)+INITIALIZERS.index(method)*.07,selected[value],
                        color=color,marker=marker,label=method if j==0 and converged else None,s=25)
        ax.set_xticks(range(3),['spatial only','independent nuisance','coupled nuisance'],rotation=25,ha='right')
        ax.set_title(title)
    axes[0].legend(fontsize=8)
    fig.suptitle('Circles: converged fits; crosses: failed convergence',fontsize=9)
    fig.tight_layout();plt.show()

## Measured interpretation and limits

Endpoint contrast converges in all nine fixed synthetic references. PC1 converges in six; soft ordinal initialization converges in three (one spatial-only and two independent-nuisance references), so its recovery averages describe only those successful seeds. Endpoint contrast retains the spatial-only/coupled results but reaches approximately 0.78 Gaussian true-position rank under independent nuisance, versus approximately 0.99 for the frozen soft segment baseline in notebook 19. Convergence therefore does not establish superior reconstruction.

The training-only anatomical starts improve the independent-nuisance starting rank. The final shared isotropic Gaussian fit can still distort the spatial coordinate in that setting. Endpoint initialization is a useful candidate, not adopted as the method merely because it makes the solver converge. The reversed-coupling failure persists, as expected for an unmodeled domain change.

No initializer is selected from query truth. A future healthy-control validation and explicit residual-variation modeling would be needed before adopting a change. The repeated physical-structure prior remains the organizing model, while these outcomes constrain what this particular mean/residual estimator can identify.